# 04 - Heston Calibration

This notebook calibrates Heston parameters to a smoothed market implied-volatility smile. The default is a single medium-dated expiry because it is compact and easy to rerun. A natural extension is multi-expiry calibration with weighted IV errors.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "heston_matplotlib_cache"))

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd

from src.calibration import calibrate_heston_to_smile, select_single_expiry_target
from src.heston import HestonParams, get_torch_device
from src.plotting import fit_iv_surface, plot_heston_fit

In [ ]:
TICKER = "AAPL"
checked_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface_checked.csv"
iv_fit_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_iv_fit.csv"

if not checked_path.exists():
    raise FileNotFoundError("Run notebook 01 first to create the checked option surface.")

df = pd.read_csv(checked_path, parse_dates=["expiry"])
spot = float((df["strike"] / df["moneyness"]).median())
rate = float(df["r"].dropna().iloc[0]) if "r" in df else 0.0
dividend = float(df["q"].dropna().iloc[0]) if "q" in df else 0.0

if iv_fit_path.exists():
    iv_fit_df = pd.read_csv(iv_fit_path, parse_dates=["expiry"])
else:
    iv_fit_df = fit_iv_surface(df, outdir=FIG_DIR)

print(f"spot={spot:.4f}, r={rate:.4f}, q={dividend:.4f}")

## Select Calibration Target

The target smile is chosen from the 3 to 9 month region where option quotes are usually less noisy than very short-dated expiries.

In [ ]:
target_expiry, maturity, strikes, target_iv = select_single_expiry_target(iv_fit_df, spot=spot)
print(f"Target expiry: {target_expiry.date()}, T={maturity:.4f}, points={len(strikes)}")

## Fit Heston Parameters

The objective minimizes mean squared error in implied-volatility space. The default path count is deliberately moderate so the notebook is runnable. Increase `n_paths` and `maxiter` for a final research run.

In [ ]:
device = get_torch_device(prefer_gpu=True)
initial = HestonParams(v0=0.04, kappa=2.0, theta=0.04, sigma=0.30, rho=-0.70)

params, objective, success, message, model_iv = calibrate_heston_to_smile(
    spot=spot,
    rate=rate,
    dividend=dividend,
    maturity=maturity,
    strikes=strikes,
    target_iv=target_iv,
    initial=initial,
    n_steps=100,
    n_paths=30_000,
    maxiter=60,
    seed=42,
    device=device,
)

print(success, message)
print(f"objective={objective:.8f}")
print(params)

In [ ]:
plot_heston_fit(strikes, target_iv, model_iv, target_expiry, outdir=FIG_DIR)

calibration_table = pd.DataFrame({
    "strike": strikes,
    "target_iv": target_iv,
    "heston_iv": model_iv,
    "iv_error": model_iv - target_iv,
})
calibration_table.head()

## Interpretation

A good calibration should match the overall smile level and skew, not every noisy point. The fitted correlation `rho` controls the leverage/skew effect, `sigma` controls volatility-of-volatility, `theta` is long-run variance, `kappa` controls mean reversion, and `v0` is the initial variance.